In [1]:
import requests
import json
import os
pw = os.environ.get("bichelppw")
import pandas as pd

In [2]:
df = pd.read_csv("https://data.colorado.gov/resource/vcag-iwy7.csv")

In [3]:
df.head()

,dataset_type,dataset_title,api_id
0,2010_standard,Census Block Groups in Colorado 2010 (https://...,93kh-r488
1,acs_old,Census Block Groups in Colorado 2011 (https://...,ic5a-quwv
2,acs_standard,Census Block Groups in Colorado 2012 (https://...,pgs5-k2ve
3,acs_standard,Census Block Groups in Colorado 2013 (https://...,muph-8jnm
4,acs_standard,Census Block Groups in Colorado 2014 (https://...,adbb-iu37


In [4]:
from sodapy import Socrata
from collections import Counter

cim_url_query = "data.colorado.gov"
census={}
with Socrata(cim_url_query, None) as client:
    datasets = client.datasets()
    for dataset in datasets:
        if dataset['owner']['display_name'] == 'Business Intelligence Center of CO':
            title=dataset["resource"]["name"]
            if 'census' in title.lower():
                census[title]=dataset

In [6]:
censusToChk={}
noCols=[]
cols={}
for title,dataset in census.items():
    values=dataset['resource']['columns_datatype']
    names=dataset['resource']['columns_name']
    w4x4 = dataset['resource']['id']
    hist = Counter(values)
    tot = sum(hist.values())
    if tot  == 0:
        noCols.append(title)
        continue
    pp = hist['Number']/tot*100
    if pp < 50.:
        print(f"{title} has {pp:.2f}% numeric columns out of {tot}")
        if tot > 10:
            censusToChk[title] = w4x4
            for n,v in zip(names,values):
                if n not in cols:
                    cols[n] = {}
                    cols[n][v] = 0
                if v not in cols[n]:
                    cols[n][v] = 0
                cols[n][v] += 1
    

Census Datasets on Colorado Information Marketplace has 0.00% numeric columns out of 3
Census Zip Codes in Colorado 2020 has 0.00% numeric columns out of 156
Census Zip Codes in Colorado 2018 has 0.00% numeric columns out of 156
Colorado Business Patterns 2019 US Census has 40.00% numeric columns out of 10
Census Zip Codes in Colorado 2019 has 0.00% numeric columns out of 156
Census Congressional Districts in Colorado 2020 has 0.00% numeric columns out of 156
Census Congressional Districts in Colorado 2018 has 0.00% numeric columns out of 156
Census Congressional Districts in Colorado 2019 has 0.00% numeric columns out of 156
Census Congressional Districts in Colorado 2022 has 0.00% numeric columns out of 156
Census Zip Codes in Colorado 2021 has 10.26% numeric columns out of 156
Census Congressional Districts in Colorado 2016 has 0.00% numeric columns out of 156
Census Congressional Districts in Colorado 2021 has 0.00% numeric columns out of 156
Census Zip Codes in Colorado 2012 has 0

In [7]:
for col,cnts in sorted(cols.items()):
    print(f"{col:>30}",end=' ')
    for t,cnt in sorted(cnts.items()):
      print(f"{t}: {cnt}",end=' ')
    print()

                      age10_14 Number: 1 Text: 14 
                      age15_19 Number: 1 Text: 14 
                      age18_24 Text: 15 
                      age20_24 Text: 15 
                      age25_29 Text: 15 
                      age30_34 Text: 15 
                      age35_39 Text: 15 
                      age40_44 Text: 15 
                      age45_49 Text: 15 
                      age50_54 Text: 15 
                      age55_59 Text: 15 
                        age5_9 Number: 1 Text: 14 
                      age60_64 Text: 15 
                      age65_69 Text: 15 
                      age70_74 Text: 15 
                      age75_79 Text: 15 
                      age80_84 Text: 15 
                       age85pl Text: 15 
                     ageless18 Text: 15 
                      ageless5 Number: 1 Text: 14 
                     armedfrcs Text: 15 
                      asian_nh Number: 1 Text: 14 
                     avghhsize Text: 15 
       

In [ ]:
  the_geom 

{'Census Zip Codes in Colorado 2020': 'ucnv-vw74',
 'Census Zip Codes in Colorado 2018': 'iuxm-ddzz',
 'Census Zip Codes in Colorado 2019': 'bjhg-77mf',
 'Census Congressional Districts in Colorado 2020': 'q56w-zzzm',
 'Census Congressional Districts in Colorado 2018': '86h6-mdkh',
 'Census Congressional Districts in Colorado 2019': 'tbmm-ubn9',
 'Census Congressional Districts in Colorado 2022': 'mv36-8akr',
 'Census Zip Codes in Colorado 2021': 'jq6r-yr82',
 'Census Congressional Districts in Colorado 2016': 'jz4n-qus2',
 'Census Congressional Districts in Colorado 2021': '7khe-gvz9',
 'Census Zip Codes in Colorado 2012': 'u47d-dmww',
 'Census Zip Codes in Colorado 2015': 'bvd7-vs7t',
 'Census Zip Codes in Colorado 2014': '82e7-gztd',
 'Census Congressional Districts in Colorado 2023': 'd4nq-ws4i',
 'Census Zip Codes in Colorado 2023': 'ap2i-xrui'}

In [ ]:
import copy
domain = "data.colorado.gov"
dataset_id = 'u47d-dmww'

username = "bic-help@xentity.com"
password = pw

url = f"https://{domain}/api/views/{dataset_id}.json"

# Step 1: Get metadata
resp = requests.get(url, auth=(username, password))
meta = resp.json()
metaO = copy.deepcopy(meta)
display(meta['columns'])
# Step 2: Modify columns
for col in meta["columns"]:
    if col["dataTypeName"] == "text" and col['name'] != "the_geom":
        # OPTIONAL: filter only specific columns
        col["dataTypeName"] = "number"


print("------------------------------\n-------------------------------\n")
#print(meta)
# Step 3: Push update
update_url = f"https://{domain}/api/views/{dataset_id}.json"

resp = requests.put(
    update_url,
    auth=(username, password),
    headers={"Content-Type": "application/json"},
    data=json.dumps(meta)
)

print(resp.status_code)
print(resp.text)

[{'id': 410818016,
  'name': 'the_geom',
  'dataTypeName': 'multipolygon',
  'description': 'GeoJSON field describing the boundaries of the area',
  'fieldName': 'the_geom',
  'position': 1,
  'renderTypeName': 'multipolygon',
  'tableColumnId': 81671524,
  'format': {}},
 {'id': 410818018,
  'name': 'zip_code',
  'dataTypeName': 'text',
  'fieldName': 'zip_code',
  'position': 2,
  'renderTypeName': 'text',
  'tableColumnId': 81671526,
  'cachedContents': {'largest': '82063',
   'non_null': '526',
   'null': '0',
   'top': [{'item': '80112', 'count': '1'},
    {'item': '81024', 'count': '1'},
    {'item': '80205', 'count': '1'},
    {'item': '80831', 'count': '1'},
    {'item': '80863', 'count': '1'},
    {'item': '81050', 'count': '1'},
    {'item': '81140', 'count': '1'},
    {'item': '81521', 'count': '1'},
    {'item': '81633', 'count': '1'},
    {'item': '81253', 'count': '1'},
    {'item': '80736', 'count': '1'},
    {'item': '80235', 'count': '1'},
    {'item': '81128', 'count'

------------------------------
-------------------------------

400
{
  "code" : "invalid_request",
  "error" : true,
  "message" : "Validation failed: Cannot change data type for column 410818018"
}



In [27]:
display(metaO['columns'])

[{'id': 410818016,
  'name': 'the_geom',
  'dataTypeName': 'multipolygon',
  'description': 'GeoJSON field describing the boundaries of the area',
  'fieldName': 'the_geom',
  'position': 1,
  'renderTypeName': 'multipolygon',
  'tableColumnId': 81671524,
  'format': {}},
 {'id': 410818018,
  'name': 'zip_code',
  'dataTypeName': 'number',
  'fieldName': 'zip_code',
  'position': 2,
  'renderTypeName': 'text',
  'tableColumnId': 81671526,
  'cachedContents': {'largest': '82063',
   'non_null': '526',
   'null': '0',
   'top': [{'item': '80112', 'count': '1'},
    {'item': '81024', 'count': '1'},
    {'item': '80205', 'count': '1'},
    {'item': '80831', 'count': '1'},
    {'item': '80863', 'count': '1'},
    {'item': '81050', 'count': '1'},
    {'item': '81140', 'count': '1'},
    {'item': '81521', 'count': '1'},
    {'item': '81633', 'count': '1'},
    {'item': '81253', 'count': '1'},
    {'item': '80736', 'count': '1'},
    {'item': '80235', 'count': '1'},
    {'item': '81128', 'coun

In [24]:
meta == metaO

True

In [25]:
type(meta)

dict